# FLEX-Med Centralized Training Benchmark

**Purpose**: Train EfficientNet-B1, EfficientNet-B2, and EfficientNet-B0 individually on their assigned Dirichlet-partitioned data splits (α=1.5) to establish centralized baselines for comparison with federated learning (FedMD) results.

**Key**: Uses the **exact same** Dirichlet partitioning, transforms, loss function (CrossEntropyLoss), optimizer (AdamW with discriminative LRs), progressive unfreezing, frozen BatchNorm, and training pipeline as the FL simulation — minus knowledge distillation and consensus aggregation.

**Model-Partition Assignment** (matches FL simulation):
- **EfficientNet-B1** → Partition 0 (3.9:1 ALL/Healthy ratio)
- **EfficientNet-B2** → Partition 1 (2.7:1 ALL/Healthy ratio)
- **EfficientNet-B0** → Partition 2 (3.1:1 ALL/Healthy ratio)

**Training Details**:
- **10 simulated FL rounds × 2 local epochs = 20 total epochs** (no early stopping, matches FL)
- **2-stage progressive unfreezing**: Phase 1 (R1-5) classifier only, Phase 2 (R6-10) 25% backbone
- **Discriminative LRs**: classifier and backbone trained at different rates
- **Frozen BatchNorm**: `set_bn_eval()` applied every epoch to prevent statistics drift

In [ ]:
# Cell 1: Setup & Imports
from google.colab import drive
drive.mount('/content/drive')

!pip install -q flwr-datasets datasets torch torchvision scikit-learn seaborn matplotlib pandas

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import models, datasets, transforms
from torchvision.models import EfficientNet_B0_Weights, EfficientNet_B1_Weights, EfficientNet_B2_Weights
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision.transforms import Compose, ToTensor, Normalize
from sklearn.metrics import roc_auc_score, confusion_matrix
from flwr_datasets.partitioner import DirichletPartitioner
from datasets import Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 2: Configuration
# ============================================================================
# Single alpha=1.5 run. Matches FL simulation exactly.
# EfficientNet-B1→P0, EfficientNet-B2→P1, EfficientNet-B0→P2
# ============================================================================

# --- Dirichlet Partitioning ---
DIRICHLET_ALPHA = 1.5

# --- Training Strategy (matches FL config exactly) ---
MINORITY_BOOST = 0.80              # Power-based rebalancing intensity
LEARNING_RATE = 0.0005             # Base learning rate (matches FL lr)
LR_DECAY = 0.95                    # Per-round LR decay
WEIGHT_DECAY = 0.02                # AdamW weight decay (backbone gets 2x)
GRAD_CLIP_NORM = 0.5               # Gradient clipping max norm
LABEL_SMOOTHING = 0.1              # CrossEntropyLoss label smoothing

# --- Progressive Unfreezing (2-stage, matches helpers.py) ---
PHASE2_START_ROUND = 6             # Rounds 1-5: head only, Rounds 6-10: 25% backbone
PHASE2_CLASSIFIER_LR_MULT = 0.6   # Classifier LR multiplier in Phase 2
PHASE2_BACKBONE_LR_MULT = 0.03    # Backbone LR multiplier in Phase 2
UNFREEZE_FRACTION = 0.25           # Fraction of last block to unfreeze in Phase 2

# --- Fixed Model -> Partition Assignment (matches FL simulation) ---
MODEL_PARTITION_MAP = {
    "efficientnet_b1": 0,          # 3.9:1 ALL/Healthy ratio
    "efficientnet_b2": 1,          # 2.7:1 ALL/Healthy ratio
    "efficientnet_b0": 2,          # 3.1:1 ALL/Healthy ratio
}
NUM_PARTITIONS = 3

# --- Fixed parameters (from FL config) ---
SEED = 42                          # Must match FL simulation seed
MIN_PARTITION_SIZE = 400           # Must match FL simulation
NUM_ROUNDS = 10                    # 10 FL rounds
LOCAL_EPOCHS_PER_ROUND = 2         # Local epochs per FL round
TOTAL_EPOCHS = NUM_ROUNDS * LOCAL_EPOCHS_PER_ROUND  # = 20 total epochs
BATCH_SIZE = 32
NUM_CLASSES = 2
IMG_SIZE = 224

# --- Dataset Paths (Google Drive) ---
TRAIN_DATASET_PATH = "/content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_local_train"
TEST_DATASET_PATH = "/content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_public_test"

# --- Model Checkpoint Save Path ---
BENCHMARK_ROOT = "/content/drive/MyDrive/College/FLEX-Med/benchmarks/centralized"
os.makedirs(BENCHMARK_ROOT, exist_ok=True)

print(f"Configuration:")
print(f"  Dirichlet alpha:  {DIRICHLET_ALPHA}")
print(f"  Model -> Partition: {MODEL_PARTITION_MAP}")
print(f"  Total models:     {len(MODEL_PARTITION_MAP)}")
print(f"  Total epochs:     {TOTAL_EPOCHS} ({NUM_ROUNDS} rounds x {LOCAL_EPOCHS_PER_ROUND} local epochs)")
print(f"  Batch size:       {BATCH_SIZE}")
print(f"  Learning rate:    {LEARNING_RATE}")
print(f"  LR decay:         {LR_DECAY} (per round)")
print(f"  Weight decay:     {WEIGHT_DECAY} (backbone: {WEIGHT_DECAY * 2})")
print(f"  Grad clip norm:   {GRAD_CLIP_NORM}")
print(f"  Label smoothing:  {LABEL_SMOOTHING}")
print(f"  Minority boost:   {MINORITY_BOOST}")
print(f"  Phase 2 start:    Round {PHASE2_START_ROUND}")
print(f"  Benchmark root:   {BENCHMARK_ROOT}")

In [ ]:
# Cell 3: Data Loading Functions
# Replicates the exact Dirichlet partitioning from task.py

# --- Transforms (identical to task.py) ---
COMMON_TRANSFORM = Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

PRIVATE_TRAIN_TRANSFORM = Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),             # Standardize resolution
    transforms.RandomHorizontalFlip(p=0.5),              # Handle arbitrary orientation
    transforms.RandomVerticalFlip(p=0.5),                # Cells aren't axis-aligned
    transforms.RandomRotation(15),                        # Slight angular variations
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),  # Prep jitter
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.0, hue=0.0),  # Staining variations
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0)),  # Microscope focus jitter
    ToTensor(),                                           # Convert to tensor (required before RandomErasing)
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),  # Slide debris
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet standards
])


class TransformOverrideSubset(torch.utils.data.Dataset):
    """Wraps a nested Subset to apply a different transform (e.g. no augmentation for val)."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
        ds = subset
        self._index_chain = []
        while isinstance(ds, Subset):
            self._index_chain.append(ds.indices)
            ds = ds.dataset
        self._root_dataset = ds

    def _resolve_index(self, idx):
        resolved = idx
        for indices in self._index_chain:
            resolved = indices[resolved]
        return resolved

    def __getitem__(self, idx):
        root_idx = self._resolve_index(idx)
        path, label = self._root_dataset.samples[root_idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

    def __len__(self):
        return len(self.subset)


def create_dirichlet_partitioner(dataset_path, num_partitions, alpha, seed, min_partition_size):
    """Create a Dirichlet partitioner (identical to task.py)."""
    full_dataset = datasets.ImageFolder(root=dataset_path, transform=None)
    image_paths = [path for path, _ in full_dataset.samples]
    labels = [label for _, label in full_dataset.samples]

    hf_dataset = Dataset.from_dict({"image_path": image_paths, "label": labels})
    partitioner = DirichletPartitioner(
        num_partitions=num_partitions, partition_by="label", alpha=alpha,
        min_partition_size=min_partition_size, self_balancing=False, shuffle=True, seed=seed
    )
    partitioner.dataset = hf_dataset
    return partitioner, full_dataset


def load_partition_dataset(partition_id, partitioner, full_dataset, batch_size, minority_boost):
    """Load a Dirichlet partition with train/val split (identical to task.py logic)."""
    partition_dataset = partitioner.load_partition(partition_id)
    partition_paths = partition_dataset["image_path"]

    path_to_idx = {path: idx for idx, (path, _) in enumerate(full_dataset.samples)}
    client_indices = [path_to_idx[path] for path in partition_paths]

    client_dataset = Subset(
        datasets.ImageFolder(root=TRAIN_DATASET_PATH, transform=PRIVATE_TRAIN_TRANSFORM),
        client_indices
    )

    # Stratified 85/15 train/val split
    class_indices = {0: [], 1: []}
    for subset_idx, global_idx in enumerate(client_indices):
        class_indices[full_dataset.targets[global_idx]].append(subset_idx)

    train_indices, val_indices = [], []
    generator = torch.Generator().manual_seed(42)

    for indices in class_indices.values():
        if not indices:
            continue
        perm = torch.randperm(len(indices), generator=generator)
        shuffled = torch.tensor(indices)[perm].tolist()
        split = int(0.85 * len(shuffled))
        train_indices.extend(shuffled[:split])
        val_indices.extend(shuffled[split:])

    train_ds = Subset(client_dataset, train_indices)
    val_ds = TransformOverrideSubset(Subset(client_dataset, val_indices), COMMON_TRANSFORM)

    # Power-based weighted random sampler (identical to task.py)
    train_targets = [full_dataset.targets[client_indices[i]] for i in train_indices]
    class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)

    # Power-based rebalancing: 0.0=natural, 0.5=sqrt, 1.0=full inverse (50/50)
    class_weights = 1.0 / np.maximum(class_counts, 1) ** minority_boost
    sample_weights = [class_weights[label] for label in train_targets]

    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    trainloader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2)
    valloader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    return trainloader, valloader, class_counts


def print_partition_stats(partitioner, full_dataset, alpha, num_partitions):
    """Print partition distribution statistics."""
    print(f"\n{'='*80}")
    print(f"PARTITION STATISTICS (alpha={alpha}, seed={SEED}, partitions={num_partitions})")
    print(f"{'='*80}")

    model_names = list(MODEL_PARTITION_MAP.keys())
    total_all, total_healthy = 0, 0
    for pid in range(num_partitions):
        partition = partitioner.load_partition(pid)
        labels = partition["label"]
        n_all = sum(l == 0 for l in labels)
        n_healthy = sum(l == 1 for l in labels)
        total = len(labels)
        all_pct = n_all / total * 100
        healthy_pct = n_healthy / total * 100
        ratio = n_all / n_healthy if n_healthy > 0 else float('inf')
        total_all += n_all
        total_healthy += n_healthy

        model_name = model_names[pid] if pid < len(model_names) else f"P{pid}"
        print(f"  Partition {pid} ({model_name}): Total={total:4d} | "
              f"ALL={n_all:4d} ({all_pct:5.1f}%) | Healthy={n_healthy:4d} ({healthy_pct:5.1f}%) | "
              f"Ratio={ratio:.1f}:1")

    overall_total = total_all + total_healthy
    overall_ratio = total_all / total_healthy if total_healthy > 0 else float('inf')
    print(f"  {'-'*74}")
    print(f"  OVERALL: Total={overall_total:4d} | Ratio={overall_ratio:.1f}:1")
    print(f"{'='*80}")


print("Data loading functions defined successfully.")

In [ ]:
# Cell 4: Model & Training Functions (matches FL pipeline exactly)

# --- Model Creation (identical to helpers.py) ---

def get_initial_dropout_rate(model_type):
    """Get default dropout rate for model architecture."""
    rates = {'efficientnet_b0': 0.40, 'efficientnet_b1': 0.40, 'efficientnet_b2': 0.35}
    return rates.get(model_type.lower(), 0.3)


def add_dropout_to_classifier(model, model_type, dropout_rate=0.3):
    """Add/update dropout layer before the final classifier head (idempotent)."""
    if isinstance(model.classifier[1], nn.Sequential) and isinstance(model.classifier[1][0], nn.Dropout):
        model.classifier[1][0].p = dropout_rate
    else:
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(in_features, NUM_CLASSES))
    return model


def get_model_by_type(model_type):
    """Create a model with pretrained weights and dropout-enhanced classifier."""
    model_type = model_type.lower()
    dropout_rate = get_initial_dropout_rate(model_type)
    model_map = {
        'efficientnet_b0': (models.efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
        'efficientnet_b1': (models.efficientnet_b1, EfficientNet_B1_Weights.DEFAULT),
        'efficientnet_b2': (models.efficientnet_b2, EfficientNet_B2_Weights.DEFAULT),
    }
    model_fn, weights = model_map[model_type]
    model = model_fn(weights=weights)
    return add_dropout_to_classifier(model, model_type, dropout_rate)


# --- Frozen BatchNorm (identical to helpers.py) ---

def set_bn_eval(m):
    """Freeze running statistics and parameters for ALL types of BatchNorm layers."""
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        m.eval()
        if m.weight is not None:
            m.weight.requires_grad = False
        if m.bias is not None:
            m.bias.requires_grad = False


# --- Progressive Unfreezing (identical to helpers.py) ---

def freeze_backbone(model):
    """Freeze all backbone layers, keep only classifier trainable."""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    return model


def unfreeze_fraction_of_last_block(model, fraction):
    """Unfreeze a fraction of the final backbone block + bridge layer."""
    # Unfreeze the bridge layer (final 1x1 conv + BN)
    for param in model.features[-1].parameters():
        param.requires_grad = True

    # Unfreeze fraction of last MBConv stage (features[-2])
    last_block = model.features[-2]
    children = list(last_block.children())
    num_children = len(children)
    num_unfreeze = int(num_children * fraction)
    if fraction > 0.0 and num_unfreeze == 0:
        num_unfreeze = 1

    unfreeze_start_idx = num_children - num_unfreeze
    for i, child in enumerate(children):
        if i >= unfreeze_start_idx:
            for param in child.parameters(recurse=True):
                param.requires_grad = True

    return model


def apply_freeze_strategy(model, simulated_round):
    """2-stage progressive unfreezing (identical to helpers.py apply_freeze_strategy)."""
    model = freeze_backbone(model)

    if simulated_round < PHASE2_START_ROUND:
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        phase = f"Phase 1 - Classifier only ({trainable:,} params)"
    else:
        model = unfreeze_fraction_of_last_block(model, UNFREEZE_FRACTION)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        phase = f"Phase 2 - 25% backbone ({trainable:,} params)"

    return model, phase


# --- Discriminative LR Optimizer (identical to task.py train()) ---

def create_optimizer(model, classifier_lr, backbone_lr):
    """Create AdamW optimizer with 4 param groups for discriminative LRs."""
    classifier_decay, classifier_no_decay = [], []
    backbone_decay, backbone_no_decay = [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        is_classifier = "classifier" in name or "fc" in name
        if is_classifier:
            if len(param.shape) == 1:
                classifier_no_decay.append(param)
            else:
                classifier_decay.append(param)
        else:
            if len(param.shape) == 1:
                backbone_no_decay.append(param)
            else:
                backbone_decay.append(param)

    return torch.optim.AdamW([
        {"params": classifier_decay, "lr": classifier_lr, "weight_decay": WEIGHT_DECAY},
        {"params": classifier_no_decay, "lr": classifier_lr, "weight_decay": 0.0},
        {"params": backbone_decay, "lr": backbone_lr, "weight_decay": WEIGHT_DECAY * 2.0},
        {"params": backbone_no_decay, "lr": backbone_lr, "weight_decay": 0.0}
    ], betas=(0.9, 0.999))


# --- Training & Validation (identical to task.py) ---

def train_one_epoch(model, trainloader, optimizer, criterion, device):
    """Train for a single epoch with frozen BN. Returns (avg_loss, accuracy)."""
    model.train()
    model.apply(set_bn_eval)  # Freeze BatchNorm statistics

    total_loss, total_batches = 0.0, 0
    correct, total = 0, 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return total_loss / max(total_batches, 1), correct / max(total, 1)


def validate(model, valloader, criterion, device):
    """Validate model. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, total_batches = 0.0, 0
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            total_batches += 1
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / max(total_batches, 1), correct / max(total, 1)


# --- Evaluation (identical metrics to task.py test()) ---

def evaluate(model, testloader, device):
    """Comprehensive evaluation with all metrics matching task.py test()."""
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, total_loss = 0, 0, 0.0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            probs = F.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().tolist())
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    accuracy = correct / total if total > 0 else 0.0
    loss = total_loss / len(testloader) if len(testloader) > 0 else 0.0
    all_preds = torch.tensor(all_preds)
    all_labels = torch.tensor(all_labels)
    all_probs = torch.tensor(all_probs)

    # Confusion matrix (class 0 = Leukemia/ALL, class 1 = Healthy)
    TP = ((all_preds == 0) & (all_labels == 0)).sum().item()
    FP = ((all_preds == 0) & (all_labels == 1)).sum().item()
    FN = ((all_preds == 1) & (all_labels == 0)).sum().item()
    TN = ((all_preds == 1) & (all_labels == 1)).sum().item()

    leukemia_acc = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    healthy_acc = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = leukemia_acc
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    specificity = healthy_acc
    class_gap = abs(healthy_acc - leukemia_acc)

    try:
        roc_auc = roc_auc_score(all_labels.numpy(), all_probs[:, 0].numpy())
    except Exception:
        roc_auc = (recall + specificity) / 2

    return {
        "loss": round(loss, 6),
        "accuracy": round(accuracy, 6),
        "precision": round(precision, 6),
        "recall": round(recall, 6),
        "f1_score": round(f1_score, 6),
        "specificity": round(specificity, 6),
        "roc_auc": round(roc_auc, 6),
        "leukemia_accuracy": round(leukemia_acc, 6),
        "healthy_accuracy": round(healthy_acc, 6),
        "class_gap": round(class_gap, 6),
        "confusion_matrix": {"TP": TP, "FP": FP, "FN": FN, "TN": TN},
        "num_samples": total,
        "all_preds": all_preds.numpy(),
        "all_labels": all_labels.numpy(),
    }


print("Functions defined successfully.")

In [ ]:
# Cell 5: Training Loop (3 EfficientNet models, single alpha=1.5)
# Simulates FL rounds: each round = freeze strategy + discriminative LRs + 2 local epochs

import random

# Deterministic seeding (matches FL client_app.py)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"{'#'*80}")
print(f"# CENTRALIZED TRAINING (Dirichlet alpha={DIRICHLET_ALPHA})")
print(f"# {NUM_ROUNDS} simulated rounds x {LOCAL_EPOCHS_PER_ROUND} local epochs = {TOTAL_EPOCHS} total epochs")
print(f"{'#'*80}")

# Create Dirichlet partitioner
partitioner, full_dataset = create_dirichlet_partitioner(
    TRAIN_DATASET_PATH, NUM_PARTITIONS, DIRICHLET_ALPHA, SEED, MIN_PARTITION_SIZE
)
print_partition_stats(partitioner, full_dataset, DIRICHLET_ALPHA, NUM_PARTITIONS)

# Loss function (identical to task.py — CrossEntropyLoss, NOT FocalLoss)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

trained_models = {}       # {model_type: model}
training_history = {}     # {model_type: history_dict}

for model_type, partition_id in MODEL_PARTITION_MAP.items():
    print(f"\n{'='*70}")
    print(f"TRAINING: {model_type.upper()} on Partition {partition_id}")
    print(f"{'='*70}")

    # Load data partition
    trainloader, valloader, class_counts = load_partition_dataset(
        partition_id, partitioner, full_dataset, BATCH_SIZE, MINORITY_BOOST
    )
    n_leukemia, n_healthy = class_counts[0], class_counts[1]
    ratio = n_leukemia / n_healthy if n_healthy > 0 else float('inf')
    print(f"Training samples: {n_leukemia + n_healthy} (ALL: {n_leukemia}, Healthy: {n_healthy}, Ratio: {ratio:.1f}:1)")
    print(f"Validation samples: {len(valloader.dataset)}")

    # Create model
    model = get_model_by_type(model_type)
    model.to(DEVICE)

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [],
        "classifier_lr": [], "backbone_lr": [], "phase": [],
        "round": []
    }

    start_time = time.time()
    epoch_counter = 0

    for sim_round in range(1, NUM_ROUNDS + 1):
        # Apply freeze strategy for this round (identical to client_app.py)
        model, phase_desc = apply_freeze_strategy(model, sim_round)

        # Compute discriminative LRs (identical to client_app.py)
        base_lr = LEARNING_RATE  # No per-round LR decay applied to base (FL doesn't decay base_lr via config)

        if sim_round < PHASE2_START_ROUND:
            classifier_lr = base_lr
            backbone_lr = 0.0
        else:
            classifier_lr = base_lr * PHASE2_CLASSIFIER_LR_MULT
            backbone_lr = base_lr * PHASE2_BACKBONE_LR_MULT

        # Create optimizer with discriminative LRs (recreated each round as freeze state changes)
        optimizer = create_optimizer(model, classifier_lr, backbone_lr)

        # Train for LOCAL_EPOCHS_PER_ROUND epochs (matches FL local training)
        for local_epoch in range(LOCAL_EPOCHS_PER_ROUND):
            epoch_counter += 1
            train_loss, train_acc = train_one_epoch(model, trainloader, optimizer, criterion, DEVICE)
            val_loss, val_acc = validate(model, valloader, criterion, DEVICE)

            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["classifier_lr"].append(classifier_lr)
            history["backbone_lr"].append(backbone_lr)
            history["phase"].append(phase_desc)
            history["round"].append(sim_round)

        # Log every round
        print(f"  Round {sim_round:2d}/{NUM_ROUNDS} | {phase_desc}")
        print(f"    CLR: {classifier_lr:.6f} | BLR: {backbone_lr:.6f} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2%} | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2%}")

    elapsed = time.time() - start_time
    final_val_acc = history["val_acc"][-1]
    best_val_acc = max(history["val_acc"])
    print(f"\nCompleted in {elapsed:.1f}s | Final Val Acc: {final_val_acc:.2%} | Best Val Acc: {best_val_acc:.2%}")

    # Save model checkpoint
    checkpoint_path = os.path.join(BENCHMARK_ROOT, f"{model_type}_partition{partition_id}.pt")
    torch.save({
        'model_type': model_type,
        'num_classes': NUM_CLASSES,
        'state_dict': model.state_dict(),
        'partition_id': partition_id,
        'alpha': DIRICHLET_ALPHA,
        'final_val_acc': final_val_acc,
    }, checkpoint_path)
    print(f"Saved: {checkpoint_path}")

    trained_models[model_type] = model
    training_history[model_type] = history

print(f"\n{'#'*80}")
print(f"ALL TRAINING COMPLETE ({len(trained_models)} models)")
print(f"{'#'*80}")

In [ ]:
# Cell 6: Evaluation on Unseen Test Set

print(f"Loading test dataset from: {TEST_DATASET_PATH}")
test_dataset = datasets.ImageFolder(root=TEST_DATASET_PATH, transform=COMMON_TRANSFORM)
testloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Test samples: {len(test_dataset)} (Classes: {test_dataset.class_to_idx})")

eval_results = {}  # {model_type: metrics_dict}

print(f"\n{'='*80}")
print(f"EVALUATING CENTRALIZED MODELS (alpha={DIRICHLET_ALPHA})")
print(f"{'='*80}")

for model_type, model in trained_models.items():
    partition_id = MODEL_PARTITION_MAP[model_type]
    print(f"\n  {model_type.upper()} (Partition {partition_id}):")
    metrics = evaluate(model, testloader, DEVICE)
    eval_results[model_type] = metrics

    cm = metrics["confusion_matrix"]
    print(f"    Accuracy:     {metrics['accuracy']:.4f}")
    print(f"    Precision:    {metrics['precision']:.4f}")
    print(f"    Recall:       {metrics['recall']:.4f}")
    print(f"    F1 Score:     {metrics['f1_score']:.4f}")
    print(f"    Specificity:  {metrics['specificity']:.4f}")
    print(f"    ROC-AUC:      {metrics['roc_auc']:.4f}")
    print(f"    Leukemia Acc: {metrics['leukemia_accuracy']:.4f}")
    print(f"    Healthy Acc:  {metrics['healthy_accuracy']:.4f}")
    print(f"    Class Gap:    {metrics['class_gap']:.4f}")
    print(f"    CM: TP={cm['TP']}, FP={cm['FP']}, FN={cm['FN']}, TN={cm['TN']}")

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}")

In [ ]:
# Cell 7: Results Table

metric_names = [
    "accuracy", "precision", "recall", "f1_score",
    "specificity", "roc_auc", "leukemia_accuracy", "healthy_accuracy",
    "class_gap", "loss"
]

table_data = {}
for model_type in MODEL_PARTITION_MAP:
    if model_type in eval_results:
        partition_id = MODEL_PARTITION_MAP[model_type]
        col_name = f"{model_type}\n(P{partition_id})"
        table_data[col_name] = [
            f"{eval_results[model_type][m]:.4f}" for m in metric_names
        ]

df = pd.DataFrame(table_data, index=[m.replace('_', ' ').title() for m in metric_names])

print(f"\n{'='*80}")
print(f"CENTRALIZED BENCHMARK RESULTS (Dirichlet alpha={DIRICHLET_ALPHA})")
print(f"{'='*80}")
print(df.to_string())
print()

# Summary row
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")

summary_rows = []
for model_type in MODEL_PARTITION_MAP:
    if model_type in eval_results:
        r = eval_results[model_type]
        summary_rows.append({
            "Model": model_type,
            "Partition": MODEL_PARTITION_MAP[model_type],
            "Accuracy": f"{r['accuracy']:.4f}",
            "Recall": f"{r['recall']:.4f}",
            "F1": f"{r['f1_score']:.4f}",
            "ROC-AUC": f"{r['roc_auc']:.4f}",
            "Class Gap": f"{r['class_gap']:.4f}",
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df

In [ ]:
# Cell 8: Confusion Matrices

class_labels = ["ALL (Leukemia)", "Healthy"]
model_list = list(MODEL_PARTITION_MAP.keys())

fig, axes = plt.subplots(1, len(model_list), figsize=(6 * len(model_list), 5))

for idx, model_type in enumerate(model_list):
    results = eval_results[model_type]
    cm = confusion_matrix(results["all_labels"], results["all_preds"])
    cm_pct = cm.astype('float') / cm.sum() * 100

    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)"

    partition_id = MODEL_PARTITION_MAP[model_type]
    sns.heatmap(
        cm, annot=annot, fmt='', cmap='Blues', ax=axes[idx],
        xticklabels=class_labels, yticklabels=class_labels,
        cbar=True, square=True, linewidths=0.5
    )
    axes[idx].set_title(
        f"{model_type.upper()} (P{partition_id})\n"
        f"Acc: {results['accuracy']:.2%}, F1: {results['f1_score']:.2%}",
        fontsize=11, fontweight='bold'
    )
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

fig.suptitle(
    f'Centralized Training - Confusion Matrices (Dirichlet alpha={DIRICHLET_ALPHA})',
    fontsize=14, fontweight='bold', y=1.05
)
plt.tight_layout()
save_path = os.path.join(BENCHMARK_ROOT, "confusion_matrices.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {save_path}")

In [ ]:
# Cell 9: Training Curves

colors = {'efficientnet_b0': '#2ca02c', 'efficientnet_b1': '#1f77b4', 'efficientnet_b2': '#ff7f0e'}

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Phase boundary: Round 5 ends at epoch 10, Round 6 starts at epoch 11
phase_boundary_epoch = (PHASE2_START_ROUND - 1) * LOCAL_EPOCHS_PER_ROUND + 0.5  # = 10.5

# --- Loss Plot ---
for model_type in MODEL_PARTITION_MAP:
    if model_type not in training_history:
        continue
    h = training_history[model_type]
    n_epochs = len(h["train_loss"])
    epochs_range = range(1, n_epochs + 1)
    color = colors.get(model_type, 'gray')
    partition_id = MODEL_PARTITION_MAP[model_type]
    label = f"{model_type} (P{partition_id})"
    axes[0].plot(epochs_range, h["train_loss"], '-', color=color, alpha=0.6, label=f"{label} train")
    axes[0].plot(epochs_range, h["val_loss"], '--', color=color, alpha=1.0, label=f"{label} val")

axes[0].axvline(x=phase_boundary_epoch, color='gray', linestyle=':', alpha=0.5, label='Phase 1/2 boundary')
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title(f'Training & Validation Loss (alpha={DIRICHLET_ALPHA})', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# --- Accuracy Plot ---
for model_type in MODEL_PARTITION_MAP:
    if model_type not in training_history:
        continue
    h = training_history[model_type]
    n_epochs = len(h["train_acc"])
    epochs_range = range(1, n_epochs + 1)
    color = colors.get(model_type, 'gray')
    partition_id = MODEL_PARTITION_MAP[model_type]
    label = f"{model_type} (P{partition_id})"
    axes[1].plot(epochs_range, [a * 100 for a in h["train_acc"]], '-', color=color, alpha=0.6, label=f"{label} train")
    axes[1].plot(epochs_range, [a * 100 for a in h["val_acc"]], '--', color=color, alpha=1.0, label=f"{label} val")

axes[1].axvline(x=phase_boundary_epoch, color='gray', linestyle=':', alpha=0.5, label='Phase 1/2 boundary')
axes[1].axhline(y=50, color='red', linestyle=':', alpha=0.3, label='Random baseline (50%)')
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_title(f'Training & Validation Accuracy (alpha={DIRICHLET_ALPHA})', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([40, 105])

# Add round markers on x-axis
round_ticks = [r * LOCAL_EPOCHS_PER_ROUND for r in range(1, NUM_ROUNDS + 1)]
ax2 = axes[1].twiny()
ax2.set_xlim(axes[1].get_xlim())
ax2.set_xticks(round_ticks)
ax2.set_xticklabels([f"R{r}" for r in range(1, NUM_ROUNDS + 1)], fontsize=8, color='gray')
ax2.set_xlabel('Simulated FL Round', fontsize=10, color='gray')

plt.tight_layout()
save_path = os.path.join(BENCHMARK_ROOT, "training_curves.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {save_path}")